In [0]:
# Databricks notebook source
# Query Delta Tables for RAG Pipeline

# MAGIC %md
# MAGIC # Query Azure Databricks Delta Tables
# MAGIC This notebook shows how to query the bronze and silver Delta tables

# Configuration
CATALOG = "main"
SCHEMA = "rag_demo"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_documents"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_chunks"

# MAGIC %md
# MAGIC ## 1. View All Documents in Bronze Table

# Query all documents
bronze_df = spark.sql(f"SELECT * FROM {BRONZE_TABLE}")
bronze_df.display()

# Count documents
doc_count = spark.sql(f"SELECT COUNT(*) as total_documents FROM {BRONZE_TABLE}").collect()[0][0]
print(f"Total documents: {doc_count}")

# MAGIC %md
# MAGIC ## 2. View All Chunks in Silver Table

# Query all chunks
silver_df = spark.sql(f"SELECT * FROM {SILVER_TABLE}")
silver_df.display()

# Count chunks
chunk_count = spark.sql(f"SELECT COUNT(*) as total_chunks FROM {SILVER_TABLE}").collect()[0][0]
print(f"Total chunks: {chunk_count}")

# MAGIC %md
# MAGIC ## 3. Search Documents by Name

# Search for specific document
document_name = "employee_handbook"  # Change this
search_results = spark.sql(f"""
SELECT 
    document_id,
    file_name,
    file_type,
    LENGTH(raw_text) as text_length,
    ingested_at
FROM {BRONZE_TABLE}
WHERE file_name LIKE '%{document_name}%'
""")
search_results.display()

# MAGIC %md
# MAGIC ## 4. Search Chunks by Content

# Search for specific content in chunks
search_text = "employee"  # Change this to search for
chunk_search = spark.sql(f"""
SELECT 
    chunk_id,
    document_id,
    file_name,
    chunk_order,
    chunk_text,
    ingested_at
FROM {SILVER_TABLE}
WHERE LOWER(chunk_text) LIKE LOWER('%{search_text}%')
ORDER BY document_id, chunk_order
""")
chunk_search.display()

# MAGIC %md
# MAGIC ## 5. Get Chunks by Document

# Get all chunks for a specific document
doc_id = "doc_001"  # Change this
doc_chunks = spark.sql(f"""
SELECT 
    chunk_id,
    chunk_order,
    chunk_text,
    file_name
FROM {SILVER_TABLE}
WHERE document_id = '{doc_id}'
ORDER BY chunk_order
""")
doc_chunks.display()

# MAGIC %md
# MAGIC ## 6. Analyze Table Statistics

# Get statistics
stats = spark.sql(f"""
SELECT
    '{BRONZE_TABLE}' as table_name,
    COUNT(*) as total_rows,
    COUNT(DISTINCT document_id) as unique_documents,
    MIN(ingested_at) as earliest_ingestion,
    MAX(ingested_at) as latest_ingestion,
    ROUND(SUM(LENGTH(raw_text))/1024/1024, 2) as total_size_mb
FROM {BRONZE_TABLE}
""")
stats.display()

# MAGIC %md
# MAGIC ## 7. SQL Queries You Can Use

# Common queries
print("=== USEFUL QUERIES ===")

# Query 1: Get latest documents
print("\n1. GET LATEST 5 DOCUMENTS:")
print(f"""
SELECT * FROM {BRONZE_TABLE}
ORDER BY ingested_at DESC
LIMIT 5
""")

# Query 2: Get chunks by file type
print("\n2. GET DOCUMENTS BY FILE TYPE:")
print(f"""
SELECT file_name, file_type, COUNT(*) as chunk_count
FROM {SILVER_TABLE}
GROUP BY file_name, file_type
""")

# Query 3: Search with regex
print("\n3. SEARCH WITH REGEX:")
print(f"""
SELECT * FROM {SILVER_TABLE}
WHERE chunk_text RLIKE 'pattern_here'
""")

# Query 4: Get document metadata
print("\n4. GET DOCUMENT METADATA:")
print(f"""
SELECT 
    document_id,
    file_name,
    file_type,
    COUNT(chunk_id) as total_chunks,
    MIN(chunk_order) as first_chunk,
    MAX(chunk_order) as last_chunk
FROM {SILVER_TABLE}
GROUP BY document_id, file_name, file_type
""")

# MAGIC %md
# MAGIC ## 8. Export Data from Delta Tables

# Export to CSV
output_path = f"/dbfs/tmp/exported_chunks.csv"
silver_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)
print(f"Data exported to: {output_path}")

# Export to Parquet
parquet_path = f"/dbfs/tmp/exported_chunks.parquet"
silver_df.write.mode("overwrite").parquet(parquet_path)
print(f"Data exported to: {parquet_path}")
